# 04 Seasonal Trends

This notebook creates a **simple winter vs summer** trend analysis for all cities and exports a table for the Streamlit Notebook Insights page.

Season definition used:
- **Winter**: November to March (`11, 12, 1, 2, 3`)
- **Summer**: April to October (`4..10`)

In [1]:
import sys
import pandas as pd
import plotly.express as px

sys.path.insert(0, "..")
from utils import load_app_ready, export_df

2026-03-28 23:57:06.295 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-28 23:57:06.302 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-28 23:57:06.302 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-28 23:57:06.303 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-28 23:57:06.303 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-28 23:57:06.304 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-28 23:57:06.304 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [2]:
df = load_app_ready()
df.shape

2026-03-28 23:57:06.311 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-28 23:57:06.311 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-28 23:57:06.315 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-28 23:57:06.316 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-28 23:57:06.317 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-28 23:57:06.318 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-28 23:57:06.318 WARNING streamlit.runtime.cachi

(15907082, 22)

In [3]:
required = ["city_name", "year", "month"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns for seasonal analysis: {missing}")

winter_months = {11, 12, 1, 2, 3}

seasonal = df.copy()
seasonal["season"] = seasonal["month"].apply(lambda m: "Winter" if int(m) in winter_months else "Summer")

agg = (
    seasonal.groupby(["city_name", "year", "season"], as_index=False)
    .agg(
        trips=("trip_id", "count"),
        avg_duration_minutes=("duration_seconds", lambda s: (s.dropna().mean() / 60) if len(s.dropna()) else None),
    )
)

agg["season"] = pd.Categorical(agg["season"], categories=["Winter", "Summer"], ordered=True)
agg = agg.sort_values(["city_name", "year", "season"]).reset_index(drop=True)
agg.head()

,city_name,year,season,trips,avg_duration_minutes
0,Bergen,2018,Winter,23611,13.898454
1,Bergen,2018,Summer,80133,15.163971
2,Bergen,2019,Winter,175918,9.383541
3,Bergen,2019,Summer,764086,11.528334
4,Bergen,2020,Winter,265487,9.225752


In [4]:
fig = px.bar(
    agg,
    x="year",
    y="trips",
    color="season",
    barmode="group",
    facet_col="city_name",
    category_orders={"season": ["Winter", "Summer"]},
    title="Seasonal Trip Trends by City (Winter vs Summer)",
    labels={"trips": "Trips", "year": "Year", "city_name": "City"},
)
fig.update_layout(height=520, legend_title_text="Season")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

In [5]:
export_name = "seasonal_trends_city_year"
export_df(export_name, agg)
print(f"Exported: {export_name}.csv")

[bridge] Exported DataFrame → c:\Users\matia\Desktop\Projects\bysykkel\Urban-Cycling\02_data\gold\notebook_exports\seasonal_trends_city_year.csv
Exported: seasonal_trends_city_year.csv


## Notes

- Exported table path is resolved from the `.env` configuration (`NOTEBOOK_EXPORTS_PATH`).
- In Streamlit, open **Notebook Insights** and press **Reload exports** to view this output.
- This table is city-ready (`city_name` column) so it can be sliced by city in future dashboard components.